In [54]:
import os
from pathlib import Path
import pandas as pd
from ollama import Client
from dotenv import load_dotenv

In [55]:
data_path = Path("../../data/raw/herd_mentality_events.csv").resolve()
processed_data_path = Path("../../data/processed/herd_mentality_events_processed_v1.csv").resolve()

load_dotenv()
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY")

In [6]:
df = pd.read_csv(data_path)
df

,Year,Event Name,Continent
0,1925,South African Rand mine strike,Africa
1,1926,Birth of Pan-African Congress,Africa
2,1927,Foundation of ANC Youth League (SA),Africa
3,1928,South African Black political conference,Africa
4,1930,Mass migration for economic reasons in Sahel,Africa
...,...,...,...
620,2021,Papua volcano eruption response,Australia/Oceania
621,2022,Fiji cyclone mass evacuations,Australia/Oceania
622,2023,Indigenous Voice to Parliament activism (Austr...,Australia/Oceania
623,2024,Pacific Islands anti-mining protests,Australia/Oceania


In [53]:
OLLAMA_API_BASE = "https://ollama.com"
OLLAMA_MODEL = "gpt-oss:120b"

# Initialize Ollama Cloud client with authentication
# NOTE: Verify your API key at https://ollama.com/settings/keys
# The API key should be a simple string, not an SSH key format
client = Client(host=OLLAMA_API_BASE, headers={'Authorization': 'Bearer ' + OLLAMA_API_KEY})

def run_ollama_chat(prompt: str,
                    model: str = OLLAMA_MODEL,
                    temperature: float = 0.1,
                    max_tokens: int | None = 1000) -> str:
    """Send a chat prompt to Ollama Cloud and return the response text."""
    response = client.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={
            "temperature": temperature,
            "num_predict": max_tokens if max_tokens else None
        }
    )
    # Ollama Client returns ChatResponse with message.content, not choices[0].message.content
    return response['message']['content']

result = run_ollama_chat("What is the capital of France?. Answer in 2 to 3 words.")
print(result)

Paris, France


In [8]:
def formating_prompt(year, event_name, continent):
    prompt_description = f"""
    You are an expert geopolitical historian AI tasked with writing concise historical descriptions of major events.

    <<INPUTS>>
    Year: {year}
    Event Name: {event_name}
    Continent: {continent}

    <<OBJECTIVE>>
    Produce a vivid, factual description of the event in between 100 to 150 words.

    <<THINKING STRATEGY>>
    1. Parse the event name, year, and continent to recall the core context and key actors.
    2. Identify the primary developments and outcomes that define the event.
    3. Prioritize the most historically significant details while staying under the word limit.
    4. Reason silently through these steps; do not expose the internal reasoning in the final response.

    <<RESPONSE CONSTRAINTS>>
    - Output must follow the template `Description: <Concise summary under 100 to 150 words>`.
    - Keep the wording neutral, factual, and free of speculation.
    - Do not include bullet points, quotes, or additional formatting beyond the required prefix.

    <<FEW-SHOT EXAMPLES>>
    Example 1
    Year: 1935
    Event Name: Italian invasion of Ethiopia (mobilizations)
    Continent: Africa
    Final Answer: Description: Italian forces invaded Ethiopia, sparking resistance led by Emperor Haile Selassie and drawing international condemnation.

    Example 2
    Year: 1940
    Event Name: Fall of France
    Continent: Europe
    Final Answer: Description: German offensives overwhelmed French defenses, leading to the capture of Paris and an armistice that split the country under occupation and the Vichy regime.

    Example 3
    Year: 1962
    Event Name: Cuban Missile Crisis
    Continent: North America
    Final Answer: Description: A tense standoff between the United States and Soviet Union over missiles in Cuba pushed the world to the brink of nuclear war before secret negotiations diffused the crisis.

    Example 4
    Year: 1931
    Event Name: Great Depression: food riots Australia
    Continent: Australia/Oceania
    Final Answer: Description: Economic collapse fueled mass protests and food riots across Australian cities as unemployed workers demanded relief and challenged government austerity.

    Example 5
    Year: 1935
    Event Name: New Guinea: mass labor protests
    Continent: Australia/Oceania
    Final Answer: Description: Indigenous laborers in New Guinea organized widespread strikes against colonial exploitation, confronting administrators and missionaries over wages and working conditions.

    Provide only the final answer in the specified format:
    Description: <Concise summary under 100 to 150 words>
    """
    return prompt_description

In [10]:
def extract_description_from_response(response_text: str, word_limit: int = 150) -> str:
    """Extract the description content from a `Description: <value>` response."""
    cleaned = response_text.strip().strip("`")
    description = None

    for line in cleaned.splitlines():
        stripped = line.strip()
        if not stripped:
            continue
        if stripped.lower().startswith("description:"):
            description = stripped.split(":", 1)[1].strip().strip('"')
            break
        if description is None:
            description = stripped

    if not description:
        return "Description unavailable"

    words = description.split()
    if len(words) > word_limit:
        description = " ".join(words[:word_limit])

    return description


def enrich_with_descriptions(dataframe: pd.DataFrame, start_idx: int = 0, end_idx: int = None) -> pd.DataFrame:
    """Enrich dataframe with descriptions. Can process a subset by specifying start_idx and end_idx."""
    if end_idx is None:
        end_idx = len(dataframe)
    
    subset_df = dataframe.iloc[start_idx:end_idx].copy()
    descriptions: list[str] = []

    for i, (idx, row) in enumerate(subset_df.iterrows()):
        year = row["Year"]
        event_name = row["Event Name"]
        continent = row["Continent"]
        prompt = formating_prompt(year, event_name, continent)
        
        try:
            response_text = run_ollama_chat(prompt)
            print(response_text)
            description = extract_description_from_response(response_text)
            descriptions.append(description)
            preview = (description[:57] + "...") if len(description) > 60 else description
            print(f"record {start_idx + i + 1} of {len(dataframe)} processed: {preview}")
        except Exception as e:
            print(f"ERROR processing record {start_idx + i + 1}: {e}")
            descriptions.append("ERROR: Failed to generate description")

    subset_df["Event Description"] = descriptions
    return subset_df


def process_with_batch_saving(df, batch_size=100):
    """
    Process data in batches, save each batch, then merge automatically.
    Automatically resumes from where it left off if interrupted.
    Returns the final merged dataframe.
    """
    import time
    from pathlib import Path
    
    batch_dir = processed_data_path.parent / "batches"
    batch_dir.mkdir(exist_ok=True)
    
    total = len(df)
    num_batches = (total + batch_size - 1) // batch_size
    
    print(f"Processing {total} records in {num_batches} batches of {batch_size}...\n")
    
    all_batches = []
    skipped_count = 0
    
    for batch_num in range(num_batches):
        start = batch_num * batch_size
        end = min(start + batch_size, total)
        batch_file = batch_dir / f"batch_{batch_num + 1:03d}.csv"
        
        # Check if batch already exists (resume capability)
        if batch_file.exists():
            print(f"⏭ Batch {batch_num + 1}/{num_batches} already exists, skipping...")
            all_batches.append(batch_file)
            skipped_count += 1
            continue
        
        print(f"Batch {batch_num + 1}/{num_batches} (records {start+1}-{end})...")
        
        try:
            batch_df = enrich_with_descriptions(df, start_idx=start, end_idx=end)
            batch_df.to_csv(batch_file, index=False)
            all_batches.append(batch_file)
            print(f"✓ Saved: {batch_file.name}\n")
        except Exception as e:
            print(f"✗ ERROR in batch {batch_num + 1}: {e}")
            print(f"  You can resume processing later - batches 1-{batch_num} are saved.\n")
            raise  # Re-raise to stop processing
        
        if batch_num < num_batches - 1:
            time.sleep(2)
    
    if skipped_count > 0:
        print(f"\nResumed: Skipped {skipped_count} already-completed batches\n")
    
    # Only merge if all batches are complete
    if len(all_batches) == num_batches:
        print("Merging all batches...")
        merged = pd.concat([pd.read_csv(f) for f in all_batches], ignore_index=True)
        merged.to_csv(processed_data_path, index=False)
        print(f"✓ Done! Final file: {processed_data_path}")
        return merged
    else:
        print(f"\n⚠ Not all batches complete yet ({len(all_batches)}/{num_batches}).")
        print("Re-run this cell to continue processing remaining batches.")
        return None

In [14]:
# Process data in batches (saves each batch and merges automatically)
# If processing stops, just re-run this cell - it will automatically resume from where it left off!
df_enriched = process_with_batch_saving(df, batch_size=100)

Processing 625 records in 7 batches of 100...

⏭ Batch 1/7 already exists, skipping...
⏭ Batch 2/7 already exists, skipping...
⏭ Batch 3/7 already exists, skipping...
⏭ Batch 4/7 already exists, skipping...
Batch 5/7 (records 401-500)...
Description: In 1991, the weakening Soviet Union triggered a cascade of independence movements across its Asian republics. Following the failed August coup, the Supreme Soviets of Kazakhstan, Kyrgyzstan, Tajikistan, Turkmenistan, and Uzbekistan declared sovereignty, held referendums, and formally proclaimed statehood between June and December. These declarations were accompanied by the establishment of new national institutions, the adoption of constitutions, and the withdrawal of Russian military and economic control. The transitions were largely peaceful, though they exposed ethnic tensions and economic challenges that would later erupt into conflicts in the region. By the end of 1991, all five Central Asian republics were internationally recognized 